In [94]:
import torch
import torch.nn as nn

测试：只取序列的最后一个时间步的输出

In [95]:
lstm_out = torch.randn(16, 150, 64)  # 16个样本，每个样本150个时间步，每个时间步64个特征
print(lstm_out)
print(lstm_out.shape)

tensor([[[-1.8707, -0.6794, -0.1703,  ...,  0.1776, -0.1101, -0.1647],
         [-0.2021,  0.7888, -0.9574,  ..., -0.9473, -0.7599,  1.2282],
         [-0.0662, -0.3341,  0.7425,  ..., -0.6443, -0.1219,  1.1780],
         ...,
         [ 0.7821,  0.2164,  0.4434,  ...,  0.3323,  0.0520,  2.1495],
         [ 0.3234,  0.4131, -1.0398,  ...,  1.1657,  0.0976,  0.4193],
         [-1.2332,  0.0785, -1.0180,  ...,  1.2127,  0.0754,  0.5740]],

        [[ 0.4871,  0.6760, -0.0452,  ..., -0.0956,  0.7223,  0.2410],
         [ 0.9383, -0.9841, -0.3879,  ..., -0.7236, -0.6700, -0.4763],
         [ 0.9720, -0.3246, -0.2011,  ..., -0.1065,  0.2752,  0.7799],
         ...,
         [ 3.3505,  0.4136,  0.7960,  ..., -0.4927,  1.6054, -0.4607],
         [ 1.4111, -0.0840,  0.4895,  ...,  0.8858, -1.4625,  0.1890],
         [ 0.0132,  0.8347,  0.5775,  ..., -1.1897, -0.3169, -0.0996]],

        [[-1.5659,  0.5573, -0.1124,  ...,  0.4236, -0.3952, -0.0877],
         [-1.6665, -1.3872, -0.1420,  ...,  0

In [96]:
out = lstm_out[:, -1, :]  # 取每个样本的最后一个时间步的输出，形状为 (16, 64)
print(out)
print(out.shape)

tensor([[-1.2332,  0.0785, -1.0180,  ...,  1.2127,  0.0754,  0.5740],
        [ 0.0132,  0.8347,  0.5775,  ..., -1.1897, -0.3169, -0.0996],
        [-1.2266, -0.0785,  2.2120,  ..., -0.2371,  1.7394, -1.3353],
        ...,
        [ 0.7356,  0.7803,  1.3421,  ...,  0.1636, -0.6233,  0.4539],
        [ 1.3955,  2.2767, -1.4854,  ...,  0.7403,  0.9316,  2.0881],
        [ 1.1818, -0.3178,  0.5499,  ..., -0.2600,  0.5431, -0.8910]])
torch.Size([16, 64])


In [97]:
dropout = nn.Dropout(p=0.5)  # 50%的概率将元素设置为0，每执行一次dropout，就有50%的元素会被设置为0
out = dropout(out)
print(out)

print(out.size(0) * out.size(1))  # 总元素数
print((out == 0).sum())  # 统计有多少个元素被设置为0

tensor([[-2.4665,  0.1571, -0.0000,  ...,  0.0000,  0.1507,  0.0000],
        [ 0.0264,  0.0000,  1.1550,  ..., -0.0000, -0.6339, -0.1992],
        [-0.0000, -0.0000,  0.0000,  ..., -0.4742,  3.4789, -2.6707],
        ...,
        [ 1.4713,  0.0000,  0.0000,  ...,  0.0000, -0.0000,  0.0000],
        [ 2.7911,  4.5535, -2.9709,  ...,  0.0000,  0.0000,  4.1762],
        [ 0.0000, -0.6357,  0.0000,  ..., -0.5200,  1.0861, -0.0000]])
1024
tensor(522)


In [98]:
# 对上一步操作返回的out进行全连接层映射  【注意】操作可叠加
fc = nn.Linear(64, 1)  # 64个特征映射到1个输出(全连接层,输出层)
out = fc(out)
out

tensor([[ 1.0132],
        [ 1.3960],
        [ 0.9772],
        [ 1.3545],
        [-0.0395],
        [ 0.0946],
        [-1.0372],
        [-0.3325],
        [ 0.1005],
        [ 0.7154],
        [-1.1141],
        [-0.5344],
        [ 0.2924],
        [ 1.8825],
        [-2.3383],
        [-0.2945]], grad_fn=<AddmmBackward0>)

In [99]:
# 对全连接层的输出应用Sigmoid激活函数  【注意】操作可叠加
sigmoid = nn.Sigmoid()
out = sigmoid(out)  # 将输出映射到(0,1)区间
print(out)
print(out.shape)

tensor([[0.7336],
        [0.8016],
        [0.7265],
        [0.7949],
        [0.4901],
        [0.5236],
        [0.2617],
        [0.4176],
        [0.5251],
        [0.6716],
        [0.2471],
        [0.3695],
        [0.5726],
        [0.8679],
        [0.0880],
        [0.4269]], grad_fn=<SigmoidBackward0>)
torch.Size([16, 1])


In [100]:
# squeeze：移除最后一个维度，形状为 (batch_size,)  【注意】操作可叠加
out = out.squeeze(1)
print(out)
print(out.shape)

tensor([0.7336, 0.8016, 0.7265, 0.7949, 0.4901, 0.5236, 0.2617, 0.4176, 0.5251,
        0.6716, 0.2471, 0.3695, 0.5726, 0.8679, 0.0880, 0.4269],
       grad_fn=<SqueezeBackward1>)
torch.Size([16])


测试测试集中的操作

In [103]:
outputs = torch.tensor([0.9538, 0.3022, 0.3022, 0.8994, 0.9532, 0.8965, 0.9298, 0.3022, 0.3329,0.3022,
0.3022, 0.9538, 0.3022, 0.3022, 0.3022, 0.3022])

(outputs > 0.5)

tensor([ True, False, False,  True,  True,  True,  True, False, False, False,
        False,  True, False, False, False, False])

In [104]:
(outputs > 0.5).float()

tensor([1., 0., 0., 1., 1., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0.])